In [3]:
import pandas as pd
import glob
import os

In [3]:
positive_controls = ['RPOB_MYCTU', 'EMBB_MYCTU', 'KATG_MYCTU', 'PNCA_MYCTU', 'GYRA_MYCTU', "GYRB_MYCTU",
       'RPOC_MYCTU', 'RSMG_MYCTU', "ETHA_MYCTU", "RS12_MYCTU", "INHA_MYCTU", "RL3_MYCTU","MMPR5_MYCTU", "ATPL_MYCTU", "TLYA_MYCTU", "DDN_MYCTU"]

In [4]:
protein_data = pd.read_csv("summary_data/01.structure_hits.csv")
protein_data = protein_data.query("uid in @positive_controls")
protein_data

,Unnamed: 0,uid,Entry,rv,set,pdb_id,pdb_chain,filtered_length
968,968,ATPL_MYCTU,P9WPS1,Rv1305,setB,2w5j,V,78.0
1106,1106,RSMG_MYCTU,P9WGW9,Rv3919c,setB,1xdz,A,202.0
1122,1122,RPOC_MYCTU,P9WGY7,Rv0668,setB,6ee8,D,1264.0
1153,1153,RS12_MYCTU,P9WH63,Rv0682,setB,5mmj,l,122.0
1212,1212,RL3_MYCTU,P9WH87,Rv0701,setB,5v7q,D,213.0
1263,1263,GYRB_MYCTU,P9WG45,Rv0005,setB,6gav,A,623.0
1276,1276,RPOB_MYCTU,P9WGY9,Rv0667,setB,6jcx,C,1138.0
1358,1358,PNCA_MYCTU,I6XD65,Rv2043c,setB,3pl1,A,185.0
1418,1418,TLYA_MYCTU,P9WJ63,Rv1694,setB,3hp7,A,257.0
1550,1550,INHA_MYCTU,P9WGR1,Rv1484,setB,2b36,C,246.0


In [5]:
## Really? GyrA and EthA didn't pass our structure filters? 

In [6]:
structure_hits = pd.read_csv("summary_data/00.structure_search_hits.csv")
structure_hits.query("uid=='GYRA_MYCTU'")

,Unnamed: 0,uid,Length_x,run_successful,num_hits,num_hits_unfiltered,percent_covered,bitscore,Length_y
5638,5638,GYRA_MYCTU,838.0,True,262.0,262.0,0.799523,0.1,838
5639,5639,GYRA_MYCTU,838.0,True,262.0,262.0,0.799523,0.2,838
5640,5640,GYRA_MYCTU,838.0,True,262.0,262.0,0.799523,0.3,838


In [7]:
structure_hits.query("uid=='ETHA_MYCTU'")

,Unnamed: 0,uid,Length_x,run_successful,num_hits,num_hits_unfiltered,percent_covered,bitscore,Length_y
6207,6207,ETHA_MYCTU,489.0,True,140.0,140.0,0.527607,0.1,489
6208,6208,ETHA_MYCTU,489.0,True,77.0,77.0,0.527607,0.2,489
6209,6209,ETHA_MYCTU,489.0,True,52.0,52.0,0.501022,0.3,489


## We neeed the list of mutations analyzed, the GEO scoring data, and the protein 3D structure to download

In [8]:
!mkdir summary_data/positive_control_download

mkdir: cannot create directory ‘summary_data/positive_control_download’: File exists


In [9]:
## Table of structure search hits

d1 = pd.read_csv("summary_data/01A.structure_search_hits_no_hits.csv")
d2 = pd.read_csv("summary_data/01B.structure_search_hits_0p90_coverage.csv")
d3 = pd.read_csv("summary_data/01C.structure_search_hits_less_than_90_percent.csv")

d = pd.concat([d1, d2, d3])
d.query("uid=='GYRA_MYCTU'")


,Unnamed: 0,uid,Length_x,run_successful,num_hits,num_hits_unfiltered,percent_covered,bitscore,Length_y,index,...,resseq_end,coord_start,coord_end,uniprot_start,uniprot_end,uniprot_id_,overlap,mapping_index,grouped_segments,percent_coverage
708,5638,GYRA_MYCTU,838.0,True,262.0,262.0,0.799523,0.1,838.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
for _,row in protein_data.iterrows():
    
    
    protein=row.uid
    uid = row.uid
    
    
    if not os.path.isfile(f"output/{protein}/all_mutations_analyzed.csv"):
        print("no mutation analyzed file")
        
    if not os.path.isfile(f"output/{protein}/G_scores.csv"):
        print("no score file")
    os.system(f"mkdir summary_data/positive_control_download/{protein}")
    os.system(f"cp output/{protein}/all_mutations_analyzed.csv summary_data/positive_control_download/{protein}/all_mutations_analyzed.csv")
    os.system(f"cp output/{protein}/G_scores.csv summary_data/positive_control_download/{protein}/G_scores.csv")
    
    os.system(f"cp filtered_distmaps/{row.Entry}* summary_data/positive_control_download/{protein}/")
    
    protein_set = row.set
    
    if protein_set=='setB':
        structure_hit = d.query("uid==@protein").iloc[0]
        bit=structure_hit.bitscore
        
        protein_structure = glob.glob(
                f"../structure_search/output/{uid}_b{bit}/compare/aux/{uid}_b{bit}_{structure_hit.pdb_id.lower()}_{structure_hit.pdb_chain}_{int(structure_hit.mapping_index)}.pdb"
            )[0]

        os.system(f"cp {protein_structure} summary_data/positive_control_download/{protein}")
        
    else: # get the alphafold hit
        entry = row.Entry
        os.system(f"cp ../alphafold/UP000001584_83332_MYCTU_v4_b70_filtered/AF-{entry}-F1-model_v4_beta70filtered.pdb summary_data/positive_control_download/{protein}")

mkdir: cannot create directory ‘summary_data/positive_control_download/ATPL_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/RSMG_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/RPOC_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/RS12_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/RL3_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/GYRB_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/RPOB_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/PNCA_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/TLYA_MYCTU’: File exists
mkdir: cannot create directory ‘summary_data/positive_control_download/INHA_MYCTU’: File exists
mkdir: cannot create directory ‘summary_d

In [1]:
!pwd

/n/data1/hms/dbmi/farhat/anna/structure_annotation/2024_11_13_GeO


In [ ]:
!../structure_search/output/RSMG_MYCTU_b0.2/compare/aux/

## Examining Cas10 and its whole deal

In [15]:
muts = pd.read_csv("output/CAS10_MYCTU/all_mutations_analyzed.csv", index_col=0)
muts.sort_values("summed").sum()

residue    104327.0
summed       1861.0
dtype: float64

In [14]:
all_muts = pd.read_csv("summary_data/02.Indels_in_analysis.csv")
all_muts.query("gene_id=='Rv2823c' and residue ==102")

,Unnamed: 0,pos,residue,gene_id,num_inframe_insertions,num_inframe_deletions,summed,SNP_type
1113,1113,3131469,102,Rv2823c,1534.0,0.0,1534.0,indel


In [12]:
all_muts = pd.read_csv("summary_data/02.all_mutations_analyzed.csv")
all_muts.query("uid=='CAS10_MYCTU'")

,Unnamed: 0,residue,summed,uid,rv,Entry
152123,4,4,3.0,CAS10_MYCTU,Rv2823c,P71629
152124,8,8,1.0,CAS10_MYCTU,Rv2823c,P71629
152125,11,11,1.0,CAS10_MYCTU,Rv2823c,P71629
152126,12,12,2.0,CAS10_MYCTU,Rv2823c,P71629
152127,13,13,1.0,CAS10_MYCTU,Rv2823c,P71629
...,...,...,...,...,...,...
152370,781,781,2.0,CAS10_MYCTU,Rv2823c,P71629
152371,785,785,1.0,CAS10_MYCTU,Rv2823c,P71629
152372,788,788,2.0,CAS10_MYCTU,Rv2823c,P71629
152373,794,794,1.0,CAS10_MYCTU,Rv2823c,P71629
